## Task 3: Model Composition (LoRA Merging)

In this task, I performed model composition by merging a LoRA fine-tuned adapter into the base GPT2 model to create a standalone improved model.

### Objective

The goal is to demonstrate how fine-tuned knowledge can be integrated directly into a base model, eliminating the need for separate adapter layers during inference.

---

### Approach

- Loaded the base GPT2 model
- Loaded the LoRA fine-tuned adapter from Task 2
- Merged the adapter weights into the base model using `merge_and_unload()`
- Compared outputs before and after merging

---

### Why This Matters

Model composition is an important technique in LLM systems because it allows:
- Efficient deployment without additional adapter overhead
- Integration of multiple learned capabilities into a single model
- Simplification of inference pipelines

In [28]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [29]:
base_model_name = "openai-community/gpt2"

base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

print("Base model loaded")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Base model loaded


In [30]:
lora_model = PeftModel.from_pretrained(base_model, "lora-gpt2")

print("LoRA model loaded")

LoRA model loaded


In [31]:
merged_model = lora_model.merge_and_unload()

print("Model merged successfully!")

Model merged successfully!


In [32]:
base_model.eval()
merged_model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [33]:
prompt = "I strongly feel that this film is"

inputs = tokenizer(prompt, return_tensors="pt")

In [34]:
output_base = base_model.generate(
    **inputs,
    max_length=50
)

print("===== BASE MODEL OUTPUT =====\n")
print(tokenizer.decode(output_base[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


===== BASE MODEL OUTPUT =====

I strongly feel that this film is a masterpiece. It is a masterpiece of cinema, and I am very proud of it. I am very proud of the fact that it is a film that has been made in the United States, and I am very


In [35]:
output_merged = merged_model.generate(
    **inputs,
    max_length=50
)

print("\n===== MERGED MODEL OUTPUT =====\n")
print(tokenizer.decode(output_merged[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



===== MERGED MODEL OUTPUT =====

I strongly feel that this film is a masterpiece. It is a masterpiece of cinema, and I am very proud of it. I am very proud of the fact that it is a film that has been made in the United States, and I am very


In [36]:
print("\n🔍 Insight:")
print("The merged model retains the fine-tuned behavior while eliminating the need for separate LoRA adapters.")
print("This demonstrates how model composition can create standalone improved models.")


🔍 Insight:
The merged model retains the fine-tuned behavior while eliminating the need for separate LoRA adapters.
This demonstrates how model composition can create standalone improved models.


## Observations

- The base and merged model outputs are very similar for the given prompt.
- This is expected due to limited fine-tuning (small dataset and only 1 epoch).
- LoRA introduces low-rank updates, which result in subtle behavioral changes rather than drastic output differences.

---

### Deeper Insight

- The merging process is still successful, as the fine-tuned parameters are integrated into the base model.
- The similarity in outputs indicates that the model requires more data or training to produce noticeable differences.
- In real-world systems, such subtle improvements accumulate across tasks and evaluations.

---

### Key Takeaway

Model composition is not always about visible output changes. It is about:
- Efficiently integrating learned improvements
- Creating deployable standalone models
- Enabling scalable LLM systems

---

### Future Improvements

To observe stronger differences:
- Increase training epochs
- Use larger or more task-specific datasets
- Test with more targeted prompts (e.g., sentiment-heavy inputs)